In [2]:
import torch
from torch import nn
from d2l import torch as d2l

# [Gradient Clipping]
#  Forward
# -> Loss 계산
# -> loss.backward()
# -> Gradient 계산 완료
# -> Gradient Clipping 
# -> optimizer.step()
# -> Parameter Update

In [ ]:
# Gradient Norm & Gradient Clipping

def model_gradient_norm(
    model: nn.Module,
) -> torch.Tensor:
    
    gradients: list[torch.Tensor] = []
    
    
    for parameter in model.parameters():
        gradient = parameter.grad

        if (
            parameter.requires_grad
            and gradient is not None
        ):
            gradients.append(gradient)    
        
        
    if not gradients:
        return torch.tensor(0.0)
    
    
    # 모든 Parameter Gradient를 하나의 거대한 Vector로 취급하여
    # Global Gradient Norm을 계산한다.
    #
    # ||g|| = sqrt(sum(g_i^2))
    squared_norms = torch.stack([
        torch.sum(gradient ** 2)
        for gradient in gradients
    ])
    
    return torch.sqrt(
        squared_norms.sum()
    )
    
    
    
@torch.no_grad()
def trainer_clip_gradients(
    self: d2l.Trainer,
    grad_clip_val: float,
    model: nn.Module,
) -> None:
    gradients: list[torch.Tensor] = []

    for parameter in model.parameters():
        gradient = parameter.grad

        if (
            parameter.requires_grad
            and gradient is not None
        ):
            gradients.append(gradient)

    if not gradients:
        return
    
    
    
    squared_norms = torch.stack([
        torch.sum(gradient ** 2)
        for gradient in gradients
    ])

    # Global Gradient Norm
    norm = torch.sqrt(
        squared_norms.sum()
    )


    # 1) norm <= theta: Gradient가 제한값 이하
    if norm <= grad_clip_val:
        return

    # 2) norm > theta: 모든 Gradient에 동일한
    # 비율을 곱해 방향을 보존하면서 크기만 축소
    scaler = grad_clip_val / norm

    for gradient in gradients:
        gradient.mul_(scaler)
    

setattr(
    d2l.Trainer,
    "clip_gradients",
    trainer_clip_gradients,
)

In [4]:
# Global Gradient Norm Calculation

gradients = [
    torch.tensor([
        [1.0, 2.0],
        [2.0, 1.0],
    ]),
    torch.tensor([
        3.0,
        4.0,
    ]),
]

squared_norms = torch.stack([
    torch.sum(gradient ** 2)
    for gradient in gradients
])

norm = torch.sqrt(
    squared_norms.sum()
)

grad_clip_val = 1.0
scale = grad_clip_val / norm

print(
    "각 Gradient의 제곱합:",
    squared_norms,
)
print(
    "Global Gradient Norm:",
    norm,
)
print(
    "Scale:",
    scale,
)

if norm > grad_clip_val:
    for gradient in gradients:
        gradient.mul_(scale)

print("\nClipped Gradients:")

for gradient in gradients:
    print(gradient)

clipped_squared_norms = torch.stack([
    torch.sum(gradient ** 2)
    for gradient in gradients
])

clipped_norm = torch.sqrt(
    clipped_squared_norms.sum()
)

print(
    "\nClipped Gradient Norm:",
    clipped_norm,
)

각 Gradient의 제곱합: tensor([10., 25.])
Global Gradient Norm: tensor(5.9161)
Scale: tensor(0.1690)

Clipped Gradients:
tensor([[0.1690, 0.3381],
        [0.3381, 0.1690]])
tensor([0.5071, 0.6761])

Clipped Gradient Norm: tensor(1.0000)
